# Change Point Detection

## 1.  Statistical Process Control (3-Sigma Rule)

In [ ]:
"""
Penjelasan tambahan untuk penelitian: Dengan menggunakan fungsi di atas, alur penentuan Change Point  
tidak lagi bergantung pada observasi visual. Jika ditanya oleh penguji/dosen mengenai metode yang digunakan, 
Anda bisa berargumen kuat dengan mengutip bahwa batas anomali getaran bearing ditentukan secara statistik matematis 
menggunakan aturan tiga sigma (mencakup 99,73% varians data sehat) yang diakui dan terbukti efektif dalam memisahkan 
sinyal sehat dan impuls cacat,. Penambahan fitur consecutive_violations pada kode (misalnya 5 detik berturut-turut) 
memastikan Change Point tidak dipicu secara tidak sengaja oleh noise acak sesaat.
"""

import numpy as np
import pandas as pd

def detect_dynamic_change_point(df, rms_column='RMS', baseline_ratio=0.1, consecutive_violations=5):
    """
    Fungsi untuk mendeteksi titik awal degradasi (Change Point / Tcp) secara dinamis
    menggunakan metode Statistical Process Control (Aturan 3-Sigma).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Dataset yang mengandung data timeseries getaran/RMS.
    rms_column : str
        Nama kolom yang berisi nilai amplitudo getaran atau RMS.
    baseline_ratio : float
        Proporsi data awal yang diasumsikan sebagai kondisi bearing sehat (default 10%).
        Bisa diganti dengan nilai absolut jika Anda tahu berapa sampel awal yang pasti sehat.
    consecutive_violations : int
        Jumlah titik berturut-turut yang harus melewati batas 3-Sigma agar
        dianggap sebagai Change Point yang valid (bukan sekadar noise/spike sementara).
        
    Returns:
    --------
    tcp_index : int
        Index baris (waktu) di mana Change Point (Tcp) terdeteksi.
    df_labeled : pandas.DataFrame
        DataFrame baru yang sudah memiliki kolom 'Label_VCD' dan batas degradasi.
    """
    
    df_labeled = df.copy()
    
    # 1. Tentukan Fase Sehat (Baseline)
    # Mengambil sekian persen data awal sebagai representasi kondisi normal (distribusi normal)
    baseline_length = int(len(df_labeled) * baseline_ratio)
    baseline_data = df_labeled[rms_column].iloc[:baseline_length]
    
    # 2. Hitung Rata-rata (Mu) dan Standar Deviasi (Sigma) dari Baseline
    # Merujuk pada asumsi paper bahwa kondisi sehat terdistribusi normal
    mu = baseline_data.mean()
    sigma = baseline_data.std()
    
    # 3. Hitung Batas Threshold 3-Sigma (Upper Control Limit)
    # P(mu - 3*sigma < X < mu + 3*sigma) = 99.73%
    ucl = mu + (3 * sigma)
    
    # 4. Deteksi Pelanggaran (Anomali yang melewati UCL)
    # Buat boolean mask di mana nilai RMS lebih besar dari batas 3-Sigma
    is_violation = df_labeled[rms_column] > ucl
    
    # 5. Cari titik di mana pelanggaran terjadi secara berturut-turut
    # Menggunakan rolling sum untuk menghitung pelanggaran konsekutif
    rolling_violations = is_violation.rolling(window=consecutive_violations).sum()
    
    # Cari index pertama di mana jumlah pelanggaran sama dengan 'consecutive_violations'
    # Kurangi dengan consecutive_violations - 1 untuk mendapatkan titik awal mula pelanggaran
    violation_indices = rolling_violations[rolling_violations == consecutive_violations].index
    
    if len(violation_indices) > 0:
        tcp_index = violation_indices - (consecutive_violations - 1)
    else:
        # Jika tidak ditemukan degradasi, asumsikan bearing tidak rusak hingga akhir
        tcp_index = len(df_labeled) - 1 
        
    # 6. Pembuatan Label VCD (Untuk LSTM-1)
    # 0 untuk sebelum Tcp, 1 untuk sesudah Tcp
    df_labeled['Label_VCD'] = 0
    df_labeled.loc[tcp_index:, 'Label_VCD'] = 1
    
    print(f"Batas Upper Control Limit (3-Sigma): {ucl:.4f}")
    print(f"Change Point (Tcp) terdeteksi pada index ke-: {tcp_index}")
    
    return tcp_index, df_labeled

# ==========================================
# CONTOH CARA PENGGUNAAN PADA ALUR ANDA:
# ==========================================
# Misal Anda memiliki dataframe 'df_bearing' dengan kolom 'Time' dan 'RMS_Vibration'
#
# tcp_index, df_hasil = detect_dynamic_change_point(df_bearing, rms_column='RMS_Vibration')
#
# Langkah selanjutnya (Untuk LSTM-2):
# df_hasil['Label_BHI'] = 1.0 # Set default awal 1.0 (Kondisi sehat)
# 
# # Hitung panjang waktu sisa setelah Tcp
# sisa_waktu = len(df_hasil) - tcp_index
# 
# # Buat penurunan BHI secara konstan dari 1.0 ke 0.0 setelah Tcp
# penurunan_bhi = np.linspace(1.0, 0.0, sisa_waktu)
# df_hasil.loc[tcp_index:, 'Label_BHI'] = penurunan_bhi